## Scenerio-**1**##

In [0]:
# Create rg
+ create resource group->rg name: adf_realtime->region: canada central->review+create->create

# create some storage account/adf/data_lake
adf_realtime->data factory->create->name(unique,ws:workspace):adf_realtime_ws-> region: east-us(default)->review+create->create
#create data lake
adf_realtime->search: storage account->create->storage account name: "adfrealtimestorage"->redundancy:LRS->✅ Enable hierarchical namespace->review+create->create
#When Blob storage will be created?/ (Diffn B/W Blob vs Data Lake Storage)
= If you unthick Enable hierarchical namespace, it will be created as Blob Storage not Data Lake. Blob is a simple storage Data Lake is a top of blob storage. 

#Now I will prepare my Data Lake/Create container in storage
"adfrealtimestorage">Containers(will be creating some folder/container here which will be act on source and one container as sink)->➕ Container->Container name: "source"->create->➕ Container->Container name: "destination"->create 
# Create Directory in data lake. Directory: is a folder
"source"->➕ Add Directory->Directory name: raw_gold_data->save


In [0]:
# Scenerio -1
Incremental Loading

# In real world incremental Load coming in SQl applying WHERE condition ,But this time it is just a CSV file source. How you will applly Azure Data factory?
⦿ Oviously it is not possible so instead of incrementally load the data we will incrementally load the file in "Source".
⦿ You cannot reload the data to "Destination" because data is already available here. How you do that?
⦿ If new file comes in "Source" that file only go to "Destination".


In [0]:
github: https://github.com/anshlambagit/ADF-RealTime-Scenarios

source/raw_gold_data->upload 1 by one file fom github so that timestamp will come 1.gold_demand_sector.csv, 2.Gold_reverse_tonnes.csv->as scenerio we only pick the 2nd data as it is latest

open adf from rg "adf_realtime_ws"->➕Linked Service->Azure Data Lake Storage Gen2->name: "ls_datalake"->select subscription->Storage Account name: "adfrealtimestorage"->test conn->creat

➕new pipeline->name: "incrementalload"->add activity>Copy Data->name: "copy_data1"->we create parametarize dataset as it can be used later also>source dataset ➕->data store:Azure Data Lake Storage Gen2, DelimitedText CSV, Name: ds_param_folderlevel, Linked Service: "ls_datalake", ->Advanced open this dataset,➕Parameters: Name:p_container (for container source),  Name: p_folder (for inside source container one folder is there raw_gold_data), again Connection File Path: Add dynamic content (we will provide paramenter instead of hard coded value) select parameter, for Directory: select p_folder, 
Name            Value
p_container     source
p_folder        raw_gold_data
->simialr for Sink:
Name            Value
p_container     destination
p_folder        raw_gold_data    
->File path type: wildcard file path: *.csv 

->Filter by last modified: (we have to design it such a way so that it will take the data from 2.17PM from that folder and that particular file of modified time)->Start time:todays date 2.00 pm->debug(now copy all the files instead of latest)->Now change startime to 2.17PM ->debug->all files here as it is UTC form(so single fill will be there instead of 2)->deactivate it now.-> we will dynamically pass this time so deactivate now.
                                                                 
Our time : UTC-4
# ------
# Our goal is incrementally load the data. Main practical Start here
solution:
    we will create get_meta_data activity on the top of the folder . Why? Because we need all the file first then we will pass that particular array to a loop which will fetch the each file, because we need to perform again 
    get_meta_data activity to fetch last_modified date. Because it need to compare and whatever date is maximum that need to put in copy_activity .

# we will change some in parameter. Craete one more parameter:
click file parameter: ➕p_file
name                        value
p_container(already there)  source
p_folder(already there).    raw_gold_data
p_file (new) 
#adf #1
get_metadata_activity->p_container:source, p_folder:raw_gold_data, p_file:Gold_reverse_tonnes.csv,->Field List: last modified->debug->you can see last modified is coming:2026-03-03;00.00.07 something like that-> delete now
get_metadata_activity->in copy_activity source->delete p_file parameter(Because we will do parametarize in folder level)->
#2
drag get_metadata_activity now(Because previous one used to check really it is fatching lastmodified time or not)->dataset "ds_param_folderlevel", p_container:source, p_folder:raw_gold_data(as of now it will give only meta data so need child item)->Field list:Child items->debug->(seeing all array, we should have 2 files available in source here)-> drag ForEach_activity connect get_metadata_activity with ForEach_activity->items: add dynamic: select 1st parameter (@activity('Get Metadata1').output.childItems)->sequentioal✅->open/expand ForEach_activity add get_metadata_activity(Because previous one used to get metadataof folder now we need each item of file so that we will get lastmodified of each file)->(so create new dataset for this get_metadata_activity)settings➕:ADLS Gen2, CSV, Name:ds_file_level,linkedservice:ls_datalake, Filepath:Advance:open dataset:parameters: 
p_container     source
p_folder        raw_gold_data
p_file          loop value(select dynamic ForEach @item().name, name available in debug output)
->Field list: Argument, LastModified(now we need to save LastModified value to a variavle using set_variable_activity so that we will compare each file)->(click outside and get out from internal get_metadata_activity click incrementalload adf) Variables:Name:"temp_max_value",:default value putting 2000-03-31T:17:17:07Z+ max_value: 2000-03-31T:17:17:07Z , type: String, Value: What will be the value  have to fetch it, so go inside ForEach add if_condition_activity -(concept: if the output of last modified of my file is greater than '2000-03-31T:17:17:07Z' value then replace this value with my lastmodified of file if not just keep at as it is)->(goal is feed copy_activity so instead of Filter by last modified put  max_value variable)
-> get_metadata_activity NEXT ForEach_activity NEXT copy_data_activity-> for each if condition: ("if get_metadata2 activity is greateer than variables("temp_max_value") then we simply replace the value with output.lastModified other wise same):  @greater(variables('temp_max_value'),activity('Get Metadata2').output.lastModified)-> if it is TRUE if_condition_activity one more set_variable_activity: Name: temp_max_value, Value: get_metadata2.lastModified 
# Text data see data # string to date format last file timestamp ex: 20250331171707
get_metadata_activity->
p_container     source
p_folder        raw_gold_data
p_file          Gold_reverse_tonnes.csv->settings Field list: Last modified->Variables:Name:V_temp->connect get_metadata_activity with set_variable_activity->set_variable_activity variable:name:V_temp, value:(dynamic using format())@formatDateTime(activity('Get Metadata1').output.lastModified,'yyyyMMddHHmmss')->ok->deug->"Error: 'filesystem' cannot be null, because it does not treating as file" To solve this error go to dataset and under Connection File path section it is blank: so take dynamic parameter  p_container, 2 p_folder, p_file Now the actual value there->Debug-> now file with date time format is coming ex:'20250331171707'-> if removed '@formatDateTime' default value will come and @activity('Get Metadata1')output.lastModified ex: '2025-03-31T17:17:07Z'